# Spatial Diagnostics: Decision-Tree Recovery from Known DGPs

`spatial_diagnostics()` runs the suite of Bayesian LM specification tests for a fitted model, and `spatial_diagnostics_decision()` walks the Koley & Bera / Anselin–Florax decision tree to recommend the next model. This notebook stress-tests both by simulating data from each cross-sectional DGP in `bayespecon.dgp`, fitting an appropriate "starting" model, and checking whether the decision tree lands on the correct generating process.

Decision-tree starting points (recap):

| Starting model | Reachable terminals |
|---|---|
| `OLS`  | OLS · SAR · SEM · SARAR |
| `SAR`  | SAR · SARAR · SDM |
| `SEM`  | SEM · SARAR · SDEM |
| `SLX`  | SLX · SDM · SDEM · MANSAR |
| `SDM`  | SDM · MANSAR |
| `SDEM` | SDEM · MANSAR |

To *recover* the true DGP we pick a starting model whose tree has the true model as a leaf. For SDM and SDEM we also try richer starting points (SAR, SEM, SLX) since the OLS path cannot reach Durbin-style terminals.

In [ ]:
import warnings

import numpy as np
import pandas as pd

from bayespecon.dgp.cross_sectional import (
    simulate_ols,
    simulate_sar,
    simulate_sdem,
    simulate_sdm,
    simulate_sem,
    simulate_slx,
)
from bayespecon.models import OLS, SAR, SDEM, SDM, SEM, SLX

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

ALPHA = 0.05  # significance for decision tree
SAMPLE_KW = dict(draws=600, tune=600, chains=2, random_seed=7, progressbar=False)

## Simulate scenarios

We generate one dataset per DGP on a 12×12 rook grid (n=144) with strong spatial signal, large sample size relative to the number of parameters, and a single non-intercept regressor (so `WX` carries one column). The seed is fixed so the notebook is reproducible.

In [ ]:
N_SIDE = 12  # 144 obs on a rook grid

# Build one shared rook-contiguity W and reuse across all DGPs (simulate_ols
# does not return W in its output dict, so we hold it explicitly here).
from bayespecon.dgp.utils import rook_grid_weights

W_dense, W_graph = rook_grid_weights(N_SIDE)

COMMON = dict(W=W_graph, seed=42, sigma=1.0)

beta = np.array([1.0, 2.0])
beta1 = np.array([1.0, 2.0])
beta2 = np.array([1.5])  # WX coefficient — large so LM-WX detects it

scenarios = {
    "OLS": simulate_ols(beta=beta, **COMMON),
    "SAR": simulate_sar(rho=0.6, beta=beta, **COMMON),
    "SEM": simulate_sem(lam=0.6, beta=beta, **COMMON),
    "SLX": simulate_slx(beta1=beta1, beta2=beta2, **COMMON),
    "SDM": simulate_sdm(rho=0.5, beta1=beta1, beta2=beta2, **COMMON),
    "SDEM": simulate_sdem(lam=0.5, beta1=beta1, beta2=beta2, **COMMON),
}

for name, d in scenarios.items():
    print(
        f"{name:5s}  n={len(d['y']):3d}  X.shape={d['X'].shape}  "
        f"true params: {list(d['params_true'].keys())}"
    )

## Helper: fit a starting model and read the decision

We wrap `X` in a DataFrame so the diagnostics tables print readable column names, then fit the chosen "starting" model on the simulated `(y, X, W)`. The decision tree is read with `format="model"` for the recommended-model string, and we also keep the diagnostic table of p-values that drove it.

In [ ]:
def to_frame(X):
    """Wrap design matrix in a DataFrame with intercept + x1, x2, ... names."""
    cols = ["intercept"] + [f"x{i}" for i in range(1, X.shape[1])]
    return pd.DataFrame(X, columns=cols)


def fit_start(start_cls, sim, **extra):
    """Fit a starting model on a simulation dict from bayespecon.dgp."""
    X = to_frame(sim["X"])
    y = sim["y"]
    model = start_cls(y=y, X=X, W=W_graph, logdet_method="eigenvalue", **extra)
    model.fit(**SAMPLE_KW)
    return model


def diagnose(model, alpha=ALPHA):
    """Return (recommended_model, diagnostics_df)."""
    diag = model.spatial_diagnostics()
    rec = model.spatial_diagnostics_decision(alpha=alpha, format="model")
    return rec, diag

## Recovery experiments

For each DGP we pick one or two natural starting models. The "expected" column lists the true generating process; the "recommended" column is what `spatial_diagnostics_decision` returned.

In [ ]:
experiments = [
    ("OLS", OLS, "OLS"),
    ("SAR", OLS, "SAR"),
    ("SEM", OLS, "SEM"),
    ("SLX", SLX, "SLX"),
    ("SDM", SAR, "SDM"),
    ("SDM", SLX, "SDM"),
    ("SDEM", SEM, "SDEM"),
    ("SDEM", SLX, "SDEM"),
]

results = []
fitted = {}
for dgp_name, start_cls, expected in experiments:
    sim = scenarios[dgp_name]
    model = fit_start(start_cls, sim)
    rec, diag = diagnose(model)
    fitted[(dgp_name, start_cls.__name__)] = (model, diag)
    results.append(
        {
            "DGP": dgp_name,
            "Starting model": start_cls.__name__,
            "Recommended": rec,
            "Expected": expected,
            "Match": "yes" if rec == expected else "no",
        }
    )

summary = pd.DataFrame(results)
summary

## Per-scenario diagnostic tables

The decision tree's choice is driven by the Bayesian LM p-values. Each table below shows the test statistics, degrees of freedom, and posterior p-values for one (DGP, starting model) experiment, alongside the chosen terminal model.

In [ ]:
for (dgp_name, start_name), (_, diag) in fitted.items():
    rec = summary.query("DGP == @dgp_name and `Starting model` == @start_name").iloc[0]
    print(
        f"\n=== DGP={dgp_name} | start={start_name} | "
        f"recommended={rec['Recommended']} ({rec['Match']} expected {rec['Expected']}) ==="
    )
    print(diag[["statistic", "df", "p_value"]].round(4).to_string())

## Visualising one decision tree

The `format="ascii"` output renders the full tree with the traversed path highlighted, which is useful when the recommendation is surprising. Below we render the tree from the SDM-from-SLX experiment.

In [ ]:
model_sdm_from_slx, _ = fitted[("SDM", "SLX")]
print(model_sdm_from_slx.spatial_diagnostics_decision(alpha=ALPHA, format="ascii"))

## Findings and caveats

**What recovered cleanly.** OLS, SAR, SEM, and SLX scenarios all returned the true generating model from their respective starting points, with the driving LM tests showing the expected pattern (e.g. only `LM-Lag` significant for SAR; only `Robust-LM-WX` patterns flat for SLX).

**What the tree missed.** The SDM and SDEM scenarios were *not* recovered as such:

- `SDM` from `SAR`: every test significant (`LM-Error`, `LM-WX`, `Robust-LM-WX`) → tree picks `SARAR`.
- `SDM` from `SLX`: both `Robust-LM-Lag-SDM` and `Robust-LM-Error-SDEM` significant → tree picks `MANSAR` (the catch-all "all four spatial channels" terminal).
- `SDEM` from `SEM` and from `SLX`: same pattern, tree picks `SARAR` / `MANSAR`.

This is the *expected* behaviour of the Koley & Bera tree at strong-signal SDM/SDEM DGPs: when both the lag and the error / WX channels look significant simultaneously, the tree escalates to the most general nested terminal rather than discriminating SDM vs. SDEM. To distinguish them, treat `SARAR`/`MANSAR` as a flag to fit both `SDM` and `SDEM` and compare with `bayes_factor_compare_models`.

**Other things to keep in mind.**

- These experiments use strong-signal parameters; weaker spatial dependence (e.g. ρ ≈ 0.1) will lead the tree toward simpler models, which is the *correct* small-sample behaviour.
- The OLS starting tree cannot reach SDM/SDEM/SLX terminals — it can only flag SAR/SEM/SARAR/OLS. Use the SAR, SEM, or SLX starting points to diagnose Durbin-style alternatives.
- The decision tree thresholds at a single `alpha`; for borderline cases inspect `spatial_diagnostics()` directly.